# 03 · Gold layer: star schema, analysis views, Power BI export

**Input:** the checked Silver tables (`IAIP\warehouse\silver\`).
**Output:** `IAIP\warehouse\gold\` (Parquet) and `IAIP\warehouse\gold\_powerbi\` (CSV files for Power BI).

### Why several fact tables?
VAHAN gives **two dimensions at a time** (state × fuel, state × category, state × norm, maker × state), never all together. So instead of one giant fact table we build **five facts at different grains that share the same dimensions**. This is called a *fact constellation* (or *galaxy*) schema:

| Fact table | Grain (one row per …) | Shares dimensions |
|---|---|---|
| `fact_registration_fuel_month` | state × **month** × fuel group × vehicle type | date, state, fuel, vehicle type |
| `fact_registration_category_year` | state × year × vehicle category × type | year, state, category, vehicle type |
| `fact_registration_norm_year` | state × year × emission norm × type | year, state, norm, vehicle type |
| `fact_registration_maker_year` | state × year × **maker** × type | year, state, maker, vehicle type |
| `fact_road_safety` | state × year | year, state |

### Gold views (written in **Spark SQL**)
`state_month_registrations`, `ev_share_state_month`, `fuel_mix_trend`, `emission_transition`, `category_growth`, `brand_share_year`, `state_safety_profile`, `state_profile_latest`.

Run top to bottom. About 3–6 minutes.

## Step 1 · Start Spark and load Silver

In [1]:
import os, sys, shutil
from pathlib import Path
from datetime import datetime

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("IAIP-03-Gold")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.ansi.enabled", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

BASE = Path.cwd().parent
WH   = BASE / "warehouse"
assert (WH / "silver" / "_quality_report").exists(), "Run 02_silver first."
RUN_TS = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# make every Silver table available to Spark SQL by name, e.g.  SELECT * FROM s_reg_state_month
SILVER = ["reg_state_month", "reg_state_category_year", "reg_state_norm_year", "reg_maker_state_year",
          "road_safety_state_year", "road_safety_national_year", "dim_state", "dim_fuel"]
for t in SILVER:
    spark.read.parquet((WH / "silver" / t).as_uri()).createOrReplaceTempView("s_" + t)

def gold(name, sql, partition_by=None):
    """Run a Spark SQL query, save the result as a Gold table, and register it for the next queries."""
    df = spark.sql(sql)
    w = df.write.mode("overwrite")
    if partition_by:
        w = w.partitionBy(*partition_by)
    w.parquet((WH / "gold" / name).as_uri())
    out = spark.read.parquet((WH / "gold" / name).as_uri())
    out.createOrReplaceTempView(name)
    print(f"gold/{name:34} {out.count():>9,} rows")
    return out

spark.sql("SELECT COUNT(*) AS silver_month_rows FROM s_reg_state_month").show()

c:\Users\Arka\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


+-----------------+
|silver_month_rows|
+-----------------+
|            35856|
+-----------------+



## Step 2 · Dimension tables

| Dimension | Key | Notes |
|---|---|---|
| `dim_date` | `date_key` = yyyymm (e.g. 202501) | month, quarter, **Indian financial year** (Apr–Mar), partial-month flag |
| `dim_year` | `year` | for the yearly facts; 2026 is partial |
| `dim_state` | `state_code` | name, region, union territory yes/no |
| `dim_fuel` | `fuel_group` | `is_ev` |
| `dim_vehicle_type` | `vehicle_type` | T = Transport (commercial), NT = Non-Transport (private) |
| `dim_vehicle_category` | `vehicle_category` | 17 categories → 6 groups |
| `dim_emission_norm` | `emission_norm` | 26 norms → 6 groups |
| `dim_maker` | `maker_key` (number) | clean name + brand |

In [2]:
gold("dim_date", """
    SELECT CAST(date_format(month_date, 'yyyyMM') AS INT)         AS date_key,
           month_date, year, month,
           date_format(month_date, 'MMM')                          AS month_name,
           quarter(month_date)                                     AS quarter,
           CASE WHEN month >= 4 THEN year ELSE year - 1 END        AS fy_start_year,
           concat('FY', CASE WHEN month >= 4 THEN year ELSE year - 1 END, '-',
                  lpad(CAST((CASE WHEN month >= 4 THEN year ELSE year - 1 END) % 100 + 1 AS STRING), 2, '0')) AS financial_year,
           bool_or(is_partial_month)                               AS is_partial_month
    FROM s_reg_state_month
    GROUP BY month_date, year, month
""")

gold("dim_year", """
    SELECT year, bool_or(is_partial_month) AS is_partial_year, COUNT(DISTINCT month) AS months_available
    FROM s_reg_state_month GROUP BY year
""")

gold("dim_state", "SELECT * FROM s_dim_state")
gold("dim_fuel", "SELECT * FROM s_dim_fuel")
gold("dim_vehicle_type", """
    SELECT * FROM VALUES ('T', 'Transport', 'Commercial: goods carriers, buses, taxis, e-rickshaws'),
                         ('NT', 'Non-Transport', 'Private: personal cars, two-wheelers, tractors')
           AS t(vehicle_type, vehicle_type_name, description)
""")
gold("dim_vehicle_category", "SELECT DISTINCT vehicle_category, category_group FROM s_reg_state_category_year")
gold("dim_emission_norm",    "SELECT DISTINCT emission_norm, norm_group FROM s_reg_state_norm_year")
gold("dim_maker", """
    SELECT DENSE_RANK() OVER (ORDER BY maker_clean) AS maker_key, maker_clean, brand
    FROM (SELECT maker_clean, MAX(brand) AS brand FROM s_reg_maker_state_year GROUP BY maker_clean) m
""")
spark.sql("SELECT * FROM dim_date ORDER BY date_key DESC LIMIT 4").show()

gold/dim_date                                  83 rows
gold/dim_year                                   7 rows
gold/dim_state                                 36 rows
gold/dim_fuel                                   6 rows
gold/dim_vehicle_type                           2 rows
gold/dim_vehicle_category                      17 rows
gold/dim_emission_norm                         26 rows
gold/dim_maker                              3,245 rows
+--------+----------+----+-----+----------+-------+-------------+--------------+----------------+
|date_key|month_date|year|month|month_name|quarter|fy_start_year|financial_year|is_partial_month|
+--------+----------+----+-----+----------+-------+-------------+--------------+----------------+
|  202611|2026-11-01|2026|   11|       Nov|      4|         2026|     FY2026-27|            true|
|  202610|2026-10-01|2026|   10|       Oct|      4|         2026|     FY2026-27|           false|
|  202609|2026-09-01|2026|    9|       Sep|      3|         2026|     

## Step 3 · Fact tables

`fact_registration_fuel_month` is saved **partitioned by year and state** (as in your proposal): each year and state gets its own folder, so a query for *"Kerala, 2025"* reads only that folder. We'll measure this in the scalability tests (E1).

In [3]:
gold("fact_registration_fuel_month", """
    SELECT CAST(date_format(month_date, 'yyyyMM') AS INT) AS date_key,
           vehicle_type, fuel_group, registrations, is_partial_month,
           year, state_code
    FROM s_reg_state_month
""", partition_by=["year", "state_code"])

gold("fact_registration_category_year", """
    SELECT year, state_code, vehicle_type, vehicle_category, registrations, is_partial_year
    FROM s_reg_state_category_year
""")

gold("fact_registration_norm_year", """
    SELECT year, state_code, vehicle_type, emission_norm, registrations, is_partial_year
    FROM s_reg_state_norm_year
""")

gold("fact_registration_maker_year", """
    SELECT m.year, m.state_code, m.vehicle_type, d.maker_key,
           SUM(m.registrations) AS registrations, BOOL_OR(m.is_partial_year) AS is_partial_year
    FROM s_reg_maker_state_year m JOIN dim_maker d ON m.maker_clean = d.maker_clean
    GROUP BY m.year, m.state_code, m.vehicle_type, d.maker_key
""")

gold("fact_road_safety", """
    SELECT year, state_code, accidents, killed, jk_includes_ladakh
    FROM s_road_safety_state_year
""")

gold/fact_registration_fuel_month          35,856 rows
gold/fact_registration_category_year        8,568 rows
gold/fact_registration_norm_year           13,104 rows
gold/fact_registration_maker_year          64,082 rows
gold/fact_road_safety                         178 rows


DataFrame[year: int, state_code: string, accidents: bigint, killed: bigint, jk_includes_ladakh: boolean]

## Step 4 · Gold views in Spark SQL

`state_code = 'IN'` rows are **All India** totals, and `vehicle_type = 'ALL'` rows are Transport + Non-Transport together, so the dashboard can show India and each state with the same table.

In [4]:
# Helper view: every fuel-month row three ways (T / NT / ALL) and at state + All-India level
spark.sql("""
CREATE OR REPLACE TEMP VIEW fm_expanded AS
WITH typed AS (
    SELECT state_code, date_key, year, vehicle_type, fuel_group, registrations FROM fact_registration_fuel_month
    UNION ALL
    SELECT state_code, date_key, year, 'ALL',        fuel_group, registrations FROM fact_registration_fuel_month
)
SELECT * FROM typed
UNION ALL
SELECT 'IN' AS state_code, date_key, year, vehicle_type, fuel_group, registrations FROM typed
""")

gold("state_month_registrations", """
    SELECT f.state_code, COALESCE(s.state_name, 'All India') AS state_name, s.region,
           d.month_date, d.year, d.month, d.financial_year, f.vehicle_type,
           SUM(f.registrations) AS registrations, d.is_partial_month
    FROM fm_expanded f JOIN dim_date d ON f.date_key = d.date_key
         LEFT JOIN dim_state s ON f.state_code = s.state_code
    GROUP BY f.state_code, s.state_name, s.region, d.month_date, d.year, d.month, d.financial_year, f.vehicle_type, d.is_partial_month
""")

gold("ev_share_state_month", """
    SELECT f.state_code, COALESCE(s.state_name, 'All India') AS state_name,
           d.month_date, d.year, d.month, f.vehicle_type,
           SUM(f.registrations)                                                  AS total_registrations,
           SUM(CASE WHEN f.fuel_group = 'EV'     THEN f.registrations ELSE 0 END) AS ev_registrations,
           SUM(CASE WHEN f.fuel_group = 'HYBRID' THEN f.registrations ELSE 0 END) AS hybrid_registrations,
           ROUND(100 * SUM(CASE WHEN f.fuel_group = 'EV' THEN f.registrations ELSE 0 END)
                 / NULLIF(SUM(f.registrations), 0), 3)                           AS ev_share_pct,
           d.is_partial_month
    FROM fm_expanded f JOIN dim_date d ON f.date_key = d.date_key
         LEFT JOIN dim_state s ON f.state_code = s.state_code
    GROUP BY f.state_code, s.state_name, d.month_date, d.year, d.month, f.vehicle_type, d.is_partial_month
""")

gold("fuel_mix_trend", """
    SELECT f.state_code, COALESCE(s.state_name, 'All India') AS state_name, f.year, f.vehicle_type, f.fuel_group,
           SUM(f.registrations) AS registrations,
           ROUND(100 * SUM(f.registrations) / SUM(SUM(f.registrations)) OVER (PARTITION BY f.state_code, f.year, f.vehicle_type), 3) AS share_pct
    FROM fm_expanded f LEFT JOIN dim_state s ON f.state_code = s.state_code
    GROUP BY f.state_code, s.state_name, f.year, f.vehicle_type, f.fuel_group
""")

gold/state_month_registrations              9,213 rows
gold/ev_share_state_month                   9,213 rows
gold/fuel_mix_trend                         4,662 rows


DataFrame[state_code: string, state_name: string, year: int, vehicle_type: string, fuel_group: string, registrations: bigint, share_pct: double]

In [5]:
gold("emission_transition", """
    WITH typed AS (
        SELECT state_code, year, vehicle_type, emission_norm, registrations FROM fact_registration_norm_year
        UNION ALL SELECT state_code, year, 'ALL', emission_norm, registrations FROM fact_registration_norm_year),
    expanded AS (SELECT * FROM typed UNION ALL SELECT 'IN', year, vehicle_type, emission_norm, registrations FROM typed)
    SELECT e.state_code, COALESCE(s.state_name, 'All India') AS state_name, e.year, e.vehicle_type, n.norm_group,
           SUM(e.registrations) AS registrations,
           ROUND(100 * SUM(e.registrations) / SUM(SUM(e.registrations)) OVER (PARTITION BY e.state_code, e.year, e.vehicle_type), 3) AS share_pct
    FROM expanded e JOIN dim_emission_norm n ON e.emission_norm = n.emission_norm
         LEFT JOIN dim_state s ON e.state_code = s.state_code
    GROUP BY e.state_code, s.state_name, e.year, e.vehicle_type, n.norm_group
""")

gold("category_growth", """
    WITH typed AS (
        SELECT state_code, year, vehicle_type, vehicle_category, registrations FROM fact_registration_category_year
        UNION ALL SELECT state_code, year, 'ALL', vehicle_category, registrations FROM fact_registration_category_year),
    expanded AS (SELECT * FROM typed UNION ALL SELECT 'IN', year, vehicle_type, vehicle_category, registrations FROM typed),
    agg AS (
        SELECT e.state_code, e.year, e.vehicle_type, c.category_group, SUM(e.registrations) AS registrations
        FROM expanded e JOIN dim_vehicle_category c ON e.vehicle_category = c.vehicle_category
        GROUP BY e.state_code, e.year, e.vehicle_type, c.category_group)
    SELECT a.state_code, COALESCE(s.state_name, 'All India') AS state_name, a.year, a.vehicle_type, a.category_group, a.registrations,
           LAG(a.registrations) OVER (PARTITION BY a.state_code, a.vehicle_type, a.category_group ORDER BY a.year) AS prev_year_registrations,
           ROUND(100 * (a.registrations - LAG(a.registrations) OVER (PARTITION BY a.state_code, a.vehicle_type, a.category_group ORDER BY a.year))
                 / NULLIF(LAG(a.registrations) OVER (PARTITION BY a.state_code, a.vehicle_type, a.category_group ORDER BY a.year), 0), 2) AS yoy_growth_pct,
           y.is_partial_year
    FROM agg a JOIN dim_year y ON a.year = y.year LEFT JOIN dim_state s ON a.state_code = s.state_code
""")

gold("brand_share_year", """
    WITH typed AS (
        SELECT year, vehicle_type, maker_key, registrations FROM fact_registration_maker_year
        UNION ALL SELECT year, 'ALL', maker_key, registrations FROM fact_registration_maker_year),
    agg AS (
        SELECT t.year, t.vehicle_type, d.brand, SUM(t.registrations) AS registrations
        FROM typed t JOIN dim_maker d ON t.maker_key = d.maker_key
        GROUP BY t.year, t.vehicle_type, d.brand)
    SELECT year, vehicle_type, brand, registrations,
           ROUND(100 * registrations / SUM(registrations) OVER (PARTITION BY year, vehicle_type), 3) AS share_pct,
           RANK() OVER (PARTITION BY year, vehicle_type ORDER BY registrations DESC) AS rank_in_year
    FROM agg
""")

gold/emission_transition                    4,662 rows
gold/category_growth                        4,662 rows
gold/brand_share_year                       1,128 rows


DataFrame[year: int, vehicle_type: string, brand: string, registrations: bigint, share_pct: double, rank_in_year: int]

### Road safety vs vehicles (`state_safety_profile`, 2020–2023)

VAHAN tells us **new** registrations, not the total number of vehicles on the road. So we report two honest ratios:
- **deaths per 10,000 new registrations** in that year
- **deaths per 10,000 vehicles registered since 2020** (a growing proxy for vehicles on the road)

For 2020, Ladakh's registrations are added to J&K, because the road-safety data still counts Ladakh inside J&K that year.

In [6]:
gold("state_safety_profile", """
    WITH reg AS (
        SELECT CASE WHEN state_code = 'LA' AND year <= 2020 THEN 'JK' ELSE state_code END AS state_code,
               year, SUM(registrations) AS new_registrations
        FROM fact_registration_fuel_month
        GROUP BY 1, 2),
    cum AS (
        SELECT state_code, year, new_registrations,
               SUM(new_registrations) OVER (PARTITION BY state_code ORDER BY year) AS registered_since_2020
        FROM reg)
    SELECT r.state_code, s.state_name, s.region, r.year, r.accidents, r.killed,
           c.new_registrations, c.registered_since_2020,
           ROUND(10000 * r.killed / NULLIF(c.new_registrations, 0), 2)     AS deaths_per_10k_new_registrations,
           ROUND(10000 * r.killed / NULLIF(c.registered_since_2020, 0), 2) AS deaths_per_10k_registered_since_2020,
           ROUND(100 * r.killed / NULLIF(r.accidents, 0), 2)               AS deaths_per_100_accidents,
           r.jk_includes_ladakh
    FROM fact_road_safety r
         JOIN cum c ON r.state_code = c.state_code AND r.year = c.year
         JOIN dim_state s ON r.state_code = s.state_code
    WHERE r.year BETWEEN 2020 AND 2023
""")

spark.sql("""
    SELECT state_name, killed, new_registrations, deaths_per_10k_new_registrations, deaths_per_100_accidents
    FROM state_safety_profile WHERE year = 2023 AND new_registrations > 100000
    ORDER BY deaths_per_10k_new_registrations DESC LIMIT 10
""").show(truncate=False)

gold/state_safety_profile                     143 rows
+--------------+------+-----------------+--------------------------------+------------------------+
|state_name    |killed|new_registrations|deaths_per_10k_new_registrations|deaths_per_100_accidents|
+--------------+------+-----------------+--------------------------------+------------------------+
|Chhattisgarh  |6166  |564517           |109.23                          |45.78                   |
|Tamil Nadu    |18347 |1826051          |100.47                          |27.3                    |
|Madhya Pradesh|13798 |1498616          |92.07                           |24.94                   |
|Andhra Pradesh|8137  |891456           |91.28                           |40.79                   |
|Telangana     |7660  |952742           |80.4                            |33.45                   |
|Odisha        |5739  |739684           |77.59                           |47.86                   |
|Rajasthan     |11762 |1525684          |77.0

### One profile per state (`state_profile_latest`)

The proposal promises *"a profile for every state and UT"*: latest full year (**2025**) registrations and growth, EV share, BS-VI share, top brand, and road deaths (latest available: **2023**).

In [7]:
gold("state_profile_latest", """
    WITH y AS (
        SELECT state_code,
               SUM(CASE WHEN year = 2025 THEN registrations END) AS reg_2025,
               SUM(CASE WHEN year = 2024 THEN registrations END) AS reg_2024,
               SUM(CASE WHEN year = 2025 AND fuel_group = 'EV' THEN registrations END) AS ev_2025,
               SUM(CASE WHEN year = 2025 AND fuel_group = 'EV' AND vehicle_type = 'NT' THEN registrations END)
                 / SUM(CASE WHEN year = 2025 AND vehicle_type = 'NT' THEN registrations END) AS ev_share_nt,
               SUM(CASE WHEN year = 2025 AND fuel_group = 'EV' AND vehicle_type = 'T' THEN registrations END)
                 / SUM(CASE WHEN year = 2025 AND vehicle_type = 'T' THEN registrations END) AS ev_share_t
        FROM fact_registration_fuel_month GROUP BY state_code),
    bs6 AS (
        SELECT f.state_code, SUM(CASE WHEN n.norm_group = 'BS-VI' THEN f.registrations ELSE 0 END) / SUM(f.registrations) AS bs6_share
        FROM fact_registration_norm_year f JOIN dim_emission_norm n ON f.emission_norm = n.emission_norm
        WHERE f.year = 2025 GROUP BY f.state_code),
    brand AS (
        SELECT state_code, brand, reg,
               ROW_NUMBER() OVER (PARTITION BY state_code ORDER BY reg DESC) AS rn,
               reg / SUM(reg) OVER (PARTITION BY state_code) AS share
        FROM (SELECT f.state_code, d.brand, SUM(f.registrations) AS reg
              FROM fact_registration_maker_year f JOIN dim_maker d ON f.maker_key = d.maker_key
              WHERE f.year = 2025 GROUP BY f.state_code, d.brand) b),
    safety AS (SELECT * FROM state_safety_profile WHERE year = 2023)
    SELECT s.state_code, s.state_name, s.region, s.is_union_territory,
           y.reg_2025, ROUND(100 * (y.reg_2025 - y.reg_2024) / NULLIF(y.reg_2024, 0), 2) AS growth_2025_pct,
           ROUND(100 * y.ev_2025 / NULLIF(y.reg_2025, 0), 2) AS ev_share_2025_pct,
           ROUND(100 * y.ev_share_nt, 2) AS ev_share_2025_private_pct,
           ROUND(100 * y.ev_share_t, 2)  AS ev_share_2025_commercial_pct,
           ROUND(100 * b6.bs6_share, 2)  AS bs6_share_2025_pct,
           br.brand AS top_brand_2025, ROUND(100 * br.share, 2) AS top_brand_share_pct,
           sf.killed AS road_deaths_2023, sf.deaths_per_10k_new_registrations AS deaths_per_10k_new_reg_2023
    FROM dim_state s
         LEFT JOIN y      ON s.state_code = y.state_code
         LEFT JOIN bs6 b6 ON s.state_code = b6.state_code
         LEFT JOIN brand br ON s.state_code = br.state_code AND br.rn = 1
         LEFT JOIN safety sf ON s.state_code = sf.state_code
""")

spark.sql("""SELECT state_name, reg_2025, growth_2025_pct, ev_share_2025_private_pct, ev_share_2025_commercial_pct,
                    bs6_share_2025_pct, top_brand_2025, road_deaths_2023
             FROM state_profile_latest ORDER BY reg_2025 DESC LIMIT 12""").show(truncate=False)

gold/state_profile_latest                      36 rows
+--------------+--------+---------------+-------------------------+----------------------------+------------------+--------------+----------------+
|state_name    |reg_2025|growth_2025_pct|ev_share_2025_private_pct|ev_share_2025_commercial_pct|bs6_share_2025_pct|top_brand_2025|road_deaths_2023|
+--------------+--------+---------------+-------------------------+----------------------------+------------------+--------------+----------------+
|Uttar Pradesh |4100623 |9.12           |3.55                     |63.0                        |86.51             |Hero MotoCorp |23652           |
|Maharashtra   |3288109 |13.88          |8.23                     |6.86                        |86.41             |Honda         |15366           |
|Tamil Nadu    |2118432 |8.45           |7.77                     |8.8                         |90.58             |TVS Motor     |18347           |
|Gujarat       |2090112 |10.04          |3.73            

## Step 5 · Gold checks: do Gold and Silver still agree?

Building views can lose or double rows by mistake (for example a bad join). So we check that the Gold totals equal the Silver totals.

In [8]:
GQC = []
def gqc(check, expected, actual, passed):
    GQC.append({"check": check, "expected": str(expected), "actual": str(actual), "passed": bool(passed), "run_ts": RUN_TS})
    print(("PASS " if passed else "FAIL ") + f"| {check} | expected {expected} | actual {actual}")

def total(sql): return spark.sql(sql).first()[0]

silver_fuel = total("SELECT SUM(registrations) FROM s_reg_state_month")
gqc("fact_registration_fuel_month = Silver", silver_fuel, total("SELECT SUM(registrations) FROM fact_registration_fuel_month"),
    silver_fuel == total("SELECT SUM(registrations) FROM fact_registration_fuel_month"))
for fact, silver in [("fact_registration_category_year", "s_reg_state_category_year"),
                     ("fact_registration_norm_year", "s_reg_state_norm_year"),
                     ("fact_registration_maker_year", "s_reg_maker_state_year")]:
    a, b = total(f"SELECT SUM(registrations) FROM {silver}"), total(f"SELECT SUM(registrations) FROM {fact}")
    gqc(f"{fact} = Silver", a, b, a == b)

india = total("SELECT SUM(registrations) FROM state_month_registrations WHERE state_code = 'IN' AND vehicle_type = 'ALL'")
gqc("view state_month_registrations: All India = sum of facts", silver_fuel, india, india == silver_fuel)
states = total("SELECT SUM(registrations) FROM state_month_registrations WHERE state_code <> 'IN' AND vehicle_type IN ('T','NT')")
gqc("view state_month_registrations: states = sum of facts", silver_fuel, states, states == silver_fuel)
bad_share = total("SELECT COUNT(*) FROM (SELECT state_code, year, vehicle_type, SUM(share_pct) s FROM fuel_mix_trend GROUP BY 1,2,3) t WHERE ABS(s - 100) > 0.05")
gqc("fuel_mix_trend: shares add to 100% in every state-year-type", 0, bad_share, bad_share == 0)
profiles = total("SELECT COUNT(*) FROM state_profile_latest WHERE reg_2025 IS NOT NULL")
gqc("state_profile_latest: a profile for every state/UT", 36, profiles, profiles == 36)

spark.createDataFrame(GQC).write.mode("overwrite").parquet((WH / "gold" / "_gold_checks").as_uri())
print(f"\nGOLD CHECKS: {sum(c['passed'] for c in GQC)} of {len(GQC)} passed")

PASS | fact_registration_fuel_month = Silver | expected 171788237 | actual 171788237
PASS | fact_registration_category_year = Silver | expected 171786708 | actual 171786708
PASS | fact_registration_norm_year = Silver | expected 171788736 | actual 171788736
PASS | fact_registration_maker_year = Silver | expected 171788736 | actual 171788736
PASS | view state_month_registrations: All India = sum of facts | expected 171788237 | actual 171788237
PASS | view state_month_registrations: states = sum of facts | expected 171788237 | actual 171788237
PASS | fuel_mix_trend: shares add to 100% in every state-year-type | expected 0 | actual 0
PASS | state_profile_latest: a profile for every state/UT | expected 36 | actual 36

GOLD CHECKS: 8 of 8 passed


## Step 6 · Export for Power BI

One CSV per table in `IAIP\warehouse\gold\_powerbi\`. In Power BI: **Get Data → Folder** (or Text/CSV) and pick these files.

In [9]:
PBI = WH / "gold" / "_powerbi"
TMP = WH / "gold" / "_tmp_csv"
PBI.mkdir(parents=True, exist_ok=True)

def export_csv(name):
    df = spark.read.parquet((WH / "gold" / name).as_uri())
    out = TMP / name
    df.coalesce(1).write.mode("overwrite").option("header", True).csv(out.as_uri())
    part = next(out.glob("part-*.csv"))
    os.replace(part, PBI / f"{name}.csv")          # one clean file per table
    print(f"_powerbi/{name}.csv   {(PBI / f'{name}.csv').stat().st_size/1e6:6.2f} MB")

EXPORT = ["dim_date", "dim_year", "dim_state", "dim_fuel", "dim_vehicle_type", "dim_vehicle_category",
          "dim_emission_norm", "dim_maker",
          "fact_registration_fuel_month", "fact_registration_category_year", "fact_registration_norm_year",
          "fact_registration_maker_year", "fact_road_safety",
          "state_month_registrations", "ev_share_state_month", "fuel_mix_trend", "emission_transition",
          "category_growth", "brand_share_year", "state_safety_profile", "state_profile_latest"]
for name in EXPORT:
    export_csv(name)
shutil.rmtree(TMP, ignore_errors=True)

_powerbi/dim_date.csv     0.00 MB
_powerbi/dim_year.csv     0.00 MB
_powerbi/dim_state.csv     0.00 MB
_powerbi/dim_fuel.csv     0.00 MB
_powerbi/dim_vehicle_type.csv     0.00 MB
_powerbi/dim_vehicle_category.csv     0.00 MB
_powerbi/dim_emission_norm.csv     0.00 MB
_powerbi/dim_maker.csv     0.19 MB
_powerbi/fact_registration_fuel_month.csv     1.21 MB
_powerbi/fact_registration_category_year.csv     0.37 MB
_powerbi/fact_registration_norm_year.csv     0.44 MB
_powerbi/fact_registration_maker_year.csv     1.60 MB
_powerbi/fact_road_safety.csv     0.00 MB
_powerbi/state_month_registrations.csv     0.60 MB
_powerbi/ev_share_state_month.csv     0.56 MB
_powerbi/fuel_mix_trend.csv     0.19 MB
_powerbi/emission_transition.csv     0.24 MB
_powerbi/category_growth.csv     0.25 MB
_powerbi/brand_share_year.csv     0.04 MB
_powerbi/state_safety_profile.csv     0.01 MB
_powerbi/state_profile_latest.csv     0.00 MB


## Step 7 · First insights

In [10]:
print("EV share of new registrations, All India (%)")
spark.sql("""
    SELECT year,
           ROUND(100 * SUM(CASE WHEN vehicle_type='NT'  THEN ev_registrations END) / SUM(CASE WHEN vehicle_type='NT'  THEN total_registrations END), 2) AS private_NT,
           ROUND(100 * SUM(CASE WHEN vehicle_type='T'   THEN ev_registrations END) / SUM(CASE WHEN vehicle_type='T'   THEN total_registrations END), 2) AS commercial_T,
           ROUND(100 * SUM(CASE WHEN vehicle_type='ALL' THEN ev_registrations END) / SUM(CASE WHEN vehicle_type='ALL' THEN total_registrations END), 2) AS all_vehicles
    FROM ev_share_state_month WHERE state_code = 'IN' GROUP BY year ORDER BY year
""").show()

print("Top 10 states by EV share in 2025 (all vehicles, states with > 1 lakh registrations)")
spark.sql("""
    SELECT state_name, SUM(total_registrations) AS registrations, ROUND(100 * SUM(ev_registrations) / SUM(total_registrations), 2) AS ev_share_pct
    FROM ev_share_state_month WHERE state_code <> 'IN' AND vehicle_type = 'ALL' AND year = 2025
    GROUP BY state_name HAVING SUM(total_registrations) > 100000 ORDER BY ev_share_pct DESC LIMIT 10
""").show(truncate=False)

print("BS-VI share of new registrations, All India (%)")
spark.sql("""SELECT year, share_pct AS bs6_share_pct FROM emission_transition
             WHERE state_code = 'IN' AND vehicle_type = 'ALL' AND norm_group = 'BS-VI' ORDER BY year""").show()

print("Top 8 brands in 2025")
spark.sql("""SELECT rank_in_year, brand, registrations, share_pct FROM brand_share_year
             WHERE year = 2025 AND vehicle_type = 'ALL' ORDER BY rank_in_year LIMIT 8""").show(truncate=False)

EV share of new registrations, All India (%)
+----+----------+------------+------------+
|year|private_NT|commercial_T|all_vehicles|
+----+----------+------------+------------+
|2020|      0.17|        7.44|        0.64|
|2021|      0.95|       12.83|        1.72|
|2022|      3.37|        19.2|        4.71|
|2023|      4.36|       23.96|        6.34|
|2024|      5.31|       26.76|        7.44|
|2025|      5.76|       28.85|        8.05|
|2026|      8.44|       29.96|       10.63|
+----+----------+------------+------------+

Top 10 states by EV share in 2025 (all vehicles, states with > 1 lakh registrations)
+--------------+-------------+------------+
|state_name    |registrations|ev_share_pct|
+--------------+-------------+------------+
|Assam         |595515       |14.3        |
|Delhi         |818006       |13.91       |
|Kerala        |888440       |12.04       |
|Karnataka     |1913041      |10.65       |
|Uttar Pradesh |4100623      |9.89        |
|Odisha        |894133       |8.8

## Done ✅

Gold is ready for **Power BI** (E5) and for the next notebooks: **04 · Kafka monthly refresh** (E2) and **05 · MLlib EV forecast + state groups** (E4).

In [ ]:
spark.stop()